# Week 08 · узнаём человека по голосу

Сегодня разбираем код speaker encoder, сравниваем **softmax** и **angular margin**,

In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Use the same torch/torchaudio pair as the trained models; GPU is not needed.
    try:
        torch_ready = all(version(name).split("+")[0] == "2.8.0" for name in ["torch", "torchaudio"])
    except PackageNotFoundError:
        torch_ready = False
    if not torch_ready:
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "torch==2.8.0", "torchaudio==2.8.0",
            "--index-url", "https://download.pytorch.org/whl/cpu",
        ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy>=1.26,<3", "matplotlib>=3.8,<4", "scikit-learn>=1.4,<2",
        "soundfile>=0.12,<0.14", "tensorboard>=2.17,<3",
        "ipywidgets>=8.1,<9", "tqdm>=4.66,<5",
    ], check=True)

    ROOT = Path("/content/week08")
    BASE_URL = "https://raw.githubusercontent.com/georgygospodinov/speech_course/2026/week08"
    for name in [
        "speaker_recognition.py", "prepare_data.py",
        "checkpoints/cross_entropy/best.ckpt", "checkpoints/angular_margin/best.ckpt",
    ]:
        target = ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            "curl", "-fLsS", "--retry", "3", "--connect-timeout", "15",
            "--output", str(target), f"{BASE_URL}/{name}",
        ], check=True)

    from google.colab import output
    output.enable_custom_widget_manager()
    print("Colab: код и чекпойнты готовы.")

In [ ]:
from pathlib import Path
import html
import sys

import IPython.display as ipd
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

if not IN_COLAB:
    ROOT = Path.cwd()
    if (ROOT / "week08").is_dir():
        ROOT = ROOT / "week08"
sys.path.insert(0, str(ROOT))
from prepare_data import download_split
from speaker_recognition import (
    Conformer, ModelParams, SpeakerDataset, collate_speakers,
    verification_pairs, verification_curve,
)

torch.set_num_threads(2)
plt.style.use("ggplot")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})
rng = np.random.default_rng(42)

## Подготовка: скачиваем dev-данные

В Colab зависимости и чекпойнты подготовлены первой ячейкой. Локально нужны
Python 3.11 и зависимости из `week08/requirements.txt`.
Обучать модели для этого ноутбука не нужно.

Данные в репозиторий не входят. Следующая ячейка при первом запуске скачает
[dev.tar](https://disk.yandex.ru/d/LKDitPbiU5XS2A) — около 266 МБ — проверит SHA256
и распакует его в `ROOT/data/dev`. В Colab это диск облачной среды, при локальном
запуске — `week08/data/dev`. Для распаковки нужно ещё около 266 МБ.
Если данные уже скачаны, ячейка использует их.

При локальном запуске можно скачать архив вручную и распаковать его в `week08/data/`.
Train-часть и логи обучения для ноутбука не нужны.

In [ ]:
DEV_DIR = ROOT / "data/dev"
if len(list(DEV_DIR.rglob("*.wav"))) != 2060:
    download_split("dev", ROOT / "data")
print(f"Dev готов: {DEV_DIR}")

## 1. Что обучали

`WAV → log-Mel → stacking subsampling → Conformer → mean/std pooling → embedding`.
При softmax обучаем классификатор дикторов. При angular margin нормируем эмбеддинги
и веса классов, увеличиваем целевой угол на `m` и применяем cross-entropy.
Верификация использует **cosine similarity эмбеддингов**, классификатор больше не нужен.

Открываем [speaker_recognition.py](https://github.com/georgygospodinov/speech_course/blob/2026/week08/speaker_recognition.py): `Conformer`,
`AngularMarginSoftmax`, цикл обучения и `evaluate`.

In [ ]:
CHECKPOINTS = {
    "softmax": ROOT / "checkpoints/cross_entropy/best.ckpt",
    "angular margin": ROOT / "checkpoints/angular_margin/best.ckpt",
}
models = {}
training_stats = {}
for name, path in CHECKPOINTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Нет {path}. Скачайте оба best.ckpt из ветки 2026 репозитория.")
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model = Conformer(ModelParams(**checkpoint["model_params"]))
    model.load_state_dict(checkpoint["model_state_dict"])
    models[name] = model.eval()
    training_stats[name] = {key: checkpoint[key] for key in ["epoch", "accuracy", "dev_eer"]}
    print(f"{name}: epoch {checkpoint['epoch']}, "
          f"train accuracy {checkpoint['accuracy']:.2%}, "
          f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 2. Сравниваем готовые модели

Сначала сравним train accuracy и dev EER выбранных эпох, сохранённые в чекпойнтах.
Почему accuracy классификации train-дикторов не заменяет проверку на новых голосах?
Далее сами посчитаем EER по всем парам dev.

In [ ]:
names = list(training_stats)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in zip(axes, ["accuracy", "dev_eer"], ["Train accuracy, %", "Dev EER, %"]):
    values = [100 * training_stats[name][key] for name in names]
    bars = ax.bar(names, values, color=["tab:blue", "tab:orange"])
    ax.bar_label(bars, fmt="%.2f")
    ax.set(title=title, ylim=(0, max(values) * 1.15))
plt.tight_layout()
plt.show()

## 3. Получаем эмбеддинги dev

Один файл — одна точка в пространстве. Папка над WAV содержит ID исходного YouTube-видео.
В этом поднаборе на диктора приходится один ролик; фон и условия записи могут помогать модели.
Поэтому результат на этих парах не оценивает перенос между разными роликами одного человека.

In [ ]:
dataset = SpeakerDataset(DEV_DIR, use_cache=True)
labels = np.array(dataset.labels)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_speakers,
)
print(f"{len(dataset)} записей, {len(np.unique(labels))} дикторов")
embeddings = {name: [] for name in models}
with torch.inference_mode():
    for _, wavs, _, lengths in tqdm(loader):
        for name, model in models.items():
            emb, _ = model(wavs, lengths)
            embeddings[name].append(torch.nn.functional.normalize(emb, dim=1))
embeddings = {name: torch.cat(parts) for name, parts in embeddings.items()}

In [ ]:
score_matrices = {name: (emb @ emb.T).numpy() for name, emb in embeddings.items()}
rows, cols, _, same = verification_pairs(embeddings["softmax"], labels)
pair_scores = {name: matrix[rows, cols] for name, matrix in score_matrices.items()}
curves = {name: verification_curve(scores, same) for name, scores in pair_scores.items()}
thresholds = {name: curve["threshold"] for name, curve in curves.items()}
print(f"Уникальных пар: {len(rows):,}; same: {same.sum():,}; different: {(~same).sum():,}")
for name, curve in curves.items():
    print(f"{name}: dev EER ≈ {curve['eer']:.2%}, threshold = {curve['threshold']:.3f}")

## 4. Матрица сходства и распределения

Блоки на диагонали — записи одного диктора. Для метрик берём только верхний треугольник
без диагонали: каждая пара один раз, без сравнения файла с самим собой.
Гистограммы нормированы отдельно, чтобы многочисленные разные пары не скрыли одинаковые.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for column, name in enumerate(models):
    image = axes[0, column].imshow(score_matrices[name], vmin=-1, vmax=1, cmap="coolwarm")
    axes[0, column].set(title=name, xlabel="clip", ylabel="clip")
    fig.colorbar(image, ax=axes[0, column])
    scores = pair_scores[name]
    axes[1, column].hist(scores[same], bins=70, density=True, alpha=.6, label="same speaker")
    axes[1, column].hist(scores[~same], bins=70, density=True, alpha=.6, label="different speakers")
    axes[1, column].axvline(thresholds[name], color="black", ls="--", label="dev EER threshold")
    axes[1, column].set(xlabel="cosine similarity", ylabel="density")
    axes[1, column].legend()
plt.tight_layout()
plt.show()

## 5. Двигаем порог

При `score >= threshold` принимаем гипотезу «один диктор».
**FAR** — доля разных пар, ошибочно принятых за одного человека.
**FRR** — доля одинаковых пар, ошибочно отклонённых.
EER здесь — среднее FAR/FRR в ближайшей эмпирической точке к их равенству;
совпадающие scores переходят через порог вместе.

Порог подбирается на dev. Здесь мы иллюстрируем калибровку;
для независимой итоговой оценки нужен отдельный test.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, curve in curves.items():
    ax.plot(curve["far"], curve["frr"], label=f"{name}: EER ≈ {curve['eer']:.2%}")
    i = curve["index"]
    ax.scatter(curve["far"][i], curve["frr"][i], s=50)
ax.plot([0, .35], [0, .35], "k--", alpha=.5)
ax.set(xlim=(0, .35), ylim=(0, .35), xlabel="FAR", ylabel="FRR", title="FAR / FRR")
ax.legend()
plt.show()

In [ ]:
def inspect_threshold(name, threshold):
    scores = pair_scores[name]
    accepted = scores >= threshold
    far = accepted[~same].mean()
    frr = (~accepted[same]).mean()
    print(f"FAR {far:.2%} ({(accepted & ~same).sum():,} false accepts); "
          f"FRR {frr:.2%} ({(~accepted & same).sum():,} false rejects)")
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.hist(scores[same], bins=70, density=True, alpha=.6, label="same")
    ax.hist(scores[~same], bins=70, density=True, alpha=.6, label="different")
    ax.axvline(threshold, color="black")
    ax.set(xlim=(-1, 1), xlabel="cosine similarity", ylabel="density")
    ax.legend()
    plt.show()

widgets.interact(
    inspect_threshold, name=list(models),
    threshold=widgets.FloatSlider(value=thresholds["angular margin"], min=-1, max=1,
                                  step=.01, continuous_update=False),
);

## 6. Слушаем пары и смотрим исходные ролики

Можно скрыть ответ и сначала угадать самим. **Обложка исходного видео может содержать
другого человека**; исходное видео показываем для контекста записи, а метку берём из датасета.
Для обложек и видео нужен интернет и доступный ролик.
Таймкоды фрагментов в этом поднаборе не сохранены, поэтому видео открывается с начала.

In [ ]:
def thumbnail_html(index):
    video_id = dataset.wav_files[index].parent.name
    source = f"https://i.ytimg.com/vi/{video_id}/hqdefault.jpg"
    return (f'<a href="https://www.youtube.com/watch?v={video_id}" target="_blank">'
            f'<img src="{source}" width="280" alt="Исходное видео {video_id}"></a>'
            f'<br>clip {index} · {html.escape(video_id)}')


def show_clip(index, video=False):
    display(ipd.HTML(thumbnail_html(index)))
    display(ipd.Audio(dataset[index][1].numpy(), rate=16_000))
    if video:
        display(ipd.YouTubeVideo(dataset.wav_files[index].parent.name, width=420, height=236))


def show_pair(i, j, video=False, reveal=True):
    if reveal:
        answer = "один диктор" if labels[i] == labels[j] else "разные дикторы"
        scores = "; ".join(f"{name}: {matrix[i, j]:.3f}" for name, matrix in score_matrices.items())
        display(ipd.Markdown(f"**{answer}** · {scores}"))
    display(ipd.HTML(f'<table><tr><td>{thumbnail_html(i)}</td><td>{thumbnail_html(j)}</td></tr></table>'))
    display(ipd.Audio(dataset[i][1].numpy(), rate=16_000))
    display(ipd.Audio(dataset[j][1].numpy(), rate=16_000))
    if video:
        for index in [i, j]:
            display(ipd.YouTubeVideo(dataset.wav_files[index].parent.name, width=420, height=236))

In [ ]:
predictions = {name: scores >= thresholds[name] for name, scores in pair_scores.items()}
correct = {name: prediction == same for name, prediction in predictions.items()}
random_pairs = np.r_[rng.choice(np.flatnonzero(same), 5, replace=False),
                     rng.choice(np.flatnonzero(~same), 5, replace=False)]


def browse_pairs(kind, name, rank, video, reveal):
    scores = pair_scores[name]
    if kind == "Случайные пары":
        ids = random_pairs
    elif kind == "False accept":
        ids = np.flatnonzero(predictions[name] & ~same)
    elif kind == "False reject":
        ids = np.flatnonzero(~predictions[name] & same)
    elif kind == "Margin исправил softmax":
        ids = np.flatnonzero(correct["angular margin"] & ~correct["softmax"])
    elif kind == "Softmax исправил margin":
        ids = np.flatnonzero(correct["softmax"] & ~correct["angular margin"])
    elif kind == "Похожие разные":
        ids = np.flatnonzero(~same)
    else:
        ids = np.flatnonzero(same)
    if not len(ids):
        print("Таких пар нет")
        return
    if kind != "Случайные пары":
        ids = ids[np.argsort(scores[ids])]
        if kind not in ["False reject", "Разные записи одного диктора"]:
            ids = ids[::-1]
    pair = ids[min(rank - 1, len(ids) - 1)]
    print(f"Пара {min(rank, len(ids))} из {len(ids):,}")
    show_pair(int(rows[pair]), int(cols[pair]), video=video, reveal=reveal)

widgets.interact(
    browse_pairs,
    kind=["Случайные пары", "False accept", "False reject", "Margin исправил softmax",
          "Softmax исправил margin", "Похожие разные", "Разные записи одного диктора"],
    name=list(models), rank=widgets.IntSlider(value=1, min=1, max=10, continuous_update=False),
    video=False, reveal=True,
);

## 7. Поиск похожих голосов

Выбираем query, слушаем ближайших соседей. Сам query исключён из выдачи.
Какие ошибки похожи на совпадение тембра, а какие — на совпадение фона или канала записи?

In [ ]:
def neighbours(query, name, top_k):
    print("Запрос")
    show_clip(query)
    scores = score_matrices[name][query].copy()
    scores[query] = -np.inf
    for index in np.argsort(scores)[-top_k:][::-1]:
        answer = "тот же диктор" if labels[index] == labels[query] else "другой диктор"
        print(f"score {scores[index]:.3f} · {answer}")
        show_clip(int(index))

widgets.interact(
    neighbours, query=widgets.IntSlider(value=0, min=0, max=len(dataset)-1, continuous_update=False),
    name=list(models), top_k=widgets.IntSlider(value=3, min=1, max=5, continuous_update=False),
);

## 8. Что изменил angular margin?

Сравниваем **те же пары** в обеих моделях. Пунктир — отдельный порог каждой модели.
Правый нижний и левый верхний квадранты — модели приняли разные решения.
Для читаемости показываем по 2 000 пар каждого типа, метрики выше посчитаны на всех.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for mask, label, color in [(~same, "different", "tab:blue"), (same, "same", "tab:orange")]:
    ids = rng.choice(np.flatnonzero(mask), min(2000, mask.sum()), replace=False)
    ax.scatter(pair_scores["softmax"][ids], pair_scores["angular margin"][ids],
               s=8, alpha=.25, label=label, color=color)
ax.axvline(thresholds["softmax"], color="black", ls="--")
ax.axhline(thresholds["angular margin"], color="black", ls="--")
ax.set(xlabel="softmax cosine", ylabel="angular margin cosine", xlim=(-1, 1), ylim=(-1, 1))
ax.legend()
plt.show()

## 9. Карта голосов

Одни и те же 12 дикторов в двух независимых PCA-проекциях. Цвет соответствует метке,
легенда использует короткие номера дикторов. Сравниваем компактность и перекрытие групп;
оси двух проекций напрямую не сопоставимы. Расстояния в 2D могут искажаться,
поэтому решения выше приняты в исходном 16-мерном пространстве.

In [ ]:
speakers = np.unique(labels)[:12]
subset = np.flatnonzero(np.isin(labels, speakers))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, emb) in zip(axes, embeddings.items()):
    xy = PCA(n_components=2).fit_transform(emb[subset].numpy())
    for number, speaker in enumerate(speakers):
        group = labels[subset] == speaker
        ax.scatter(xy[group, 0], xy[group, 1], s=18, label=str(number),
                   color=plt.get_cmap("tab20")(number))
    ax.set(title=name, xlabel="PC 1", ylabel="PC 2")
axes[-1].legend(title="speaker", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

## 10. Сколько речи нужно?

Берём две записи одного диктора. Одну оставляем целиком, вторую обрезаем.
Это эксперимент с одной парой, а не оценка качества на всех длительностях.
Прослушиваем короткий фрагмент и проверяем, насколько уверены сами.

In [ ]:
def duration_experiment(reference=0, query=1):
    wav = dataset[query][1]
    durations = [s for s in [.25, .5, 1., 2., 3., 4.] if int(s * 16_000) <= len(wav)]
    fig, ax = plt.subplots(figsize=(8, 4))
    with torch.inference_mode():
        for name, model in models.items():
            similarities = []
            for seconds in durations:
                short_emb, _ = model(wav[:int(seconds * 16_000)].unsqueeze(0))
                short_emb = torch.nn.functional.normalize(short_emb, dim=1)
                similarities.append(float(short_emb[0] @ embeddings[name][reference]))
            ax.plot(durations, similarities, "o-", label=name)
            ax.axhline(thresholds[name], ls="--", alpha=.4)
    ax.set(xlabel="query duration, seconds", ylabel="cosine similarity")
    ax.legend()
    plt.show()
    display(ipd.Audio(wav[:8000].numpy(), rate=16_000))

duration_experiment()

Что обсудить в конце: почему хорошие train accuracy и dev EER — разные вещи;
как порог меняет тип ошибок; зачем нужны новые дикторы и независимый test;
какую роль играют длительность, шум и условия записи.

Можно вернуться к карточкам пар, скрыть ответы и сравнить решения аудитории с моделями.